## 2107540 Data Science for Environmental and Sustainability
## Lecture 3 : SQL Database Commands
### Done by : 6730084521 Chatrphol Ovanonchai

## Lab Setup

In [2]:
import pandas as pd 
import matplotlib.pyplot as plt 
import sqlite3
import seaborn as sns

In [3]:
import os

data_path = "../Data"
figure_path = "../figures"
database_path = "../Database"
os.makedirs(figure_path , exist_ok = True) # make directory 

## create database path & connector
db_path = f"{database_path}/esg_database_lab.db"
conn = sqlite3.connect("../Database/esg_database_lab.db")

## write run_sql command to use sql command
def run_sql(query_cmd , connector) : 
    """Run a SQL query and return the result as a DataFrame."""
    return pd.read_sql_query(sql = query_cmd , con = connector)

def read_and_load_csv_to_db(file_path , connector , table_name , encoding) : 
    # read csv (with encoding)
    df = pd.read_csv(file_path , encoding = encoding)
    # load csv to db
    df.to_sql(name = table_name , con = connector , if_exists = "replace" , index = False) 
    
def execute_sql(command , connector = conn) : 
    connector.execute(command) 
    connector.commit() # to change data 
    
## run_sql wouldn't change data , while execute_sql would change data forever

In [5]:
## Create Database
# run_sql("CREATE DATABASE esg_database")

## SQLITE can't create new db , so using connect instead
esg_lab_conn = sqlite3.connect("../Database/esg_database_lab.db")  # creates the file if it doesn't exist

In [6]:
## open foreign_keys
esg_lab_conn.execute("PRAGMA foreign_keys = ON")

## Import Data

In [7]:
## using read_and_load_csv_to_db function
read_and_load_csv_to_db(file_path = "../Data/company_info.csv" , connector = esg_lab_conn , 
                        table_name = "company_info" , encoding = "utf-8-sig")

read_and_load_csv_to_db(file_path = "../Data/company_perf.csv" , connector = esg_lab_conn , 
                        table_name = "company_perf" , encoding = "utf-8-sig")

In [11]:
## see data 
run_sql("""
SELECT * FROM company_perf
""" , connector = esg_lab_conn) 

,performance_id,company_id,year,revenue,profit_margin,market_cap,growth_rate,esg_overall,esg_environmental,esg_social,esg_governance,carbon_emissions,water_usage,energy_consumption
0,1,1,2015,459.2,6.0,337.5,NaN,57.0,60.7,33.5,76.8,35577.4,17788.7,71154.7
1,2,1,2016,473.8,4.6,366.6,3.2,56.7,58.9,32.8,78.5,37314.7,18657.4,74629.4
2,3,1,2017,564.9,5.2,313.4,19.2,56.5,57.6,34.0,77.8,45006.4,22503.2,90012.9
3,4,1,2018,558.4,4.3,283.0,-1.1,58.0,62.3,33.4,78.3,42650.1,21325.1,85300.2
4,5,1,2019,554.5,4.9,538.1,-0.7,56.6,63.7,30.0,76.1,41799.4,20899.7,83598.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10995,10996,1000,2021,3419.2,21.7,14416.7,0.4,52.2,67.2,46.3,43.1,498875.8,623594.7,2494378.8
10996,10997,1000,2022,3518.8,21.4,8078.9,2.9,51.9,68.7,44.7,42.3,506013.7,632517.1,2530068.4
10997,10998,1000,2023,3417.1,17.1,13263.4,-2.9,52.9,72.4,43.1,43.2,473561.5,591951.9,2367807.7
10998,10999,1000,2024,3183.2,15.7,6960.1,-6.8,52.0,72.4,40.5,42.9,441310.9,551638.6,2206554.3


### Question 1
Find all performance records from 2025 where carbon_emissions are greater than 500.

Display:
- company_id
- year
- carbon_emissions
- water_usage

In [9]:
run_sql("""
SELECT company_id , year , carbon_emissions , water_usage
FROM company_perf 
WHERE carbon_emissions > 500
    AND year = 2025
""" , connector = esg_lab_conn) 

,company_id,year,carbon_emissions,water_usage
0,1,2025,51805.8,25902.9
1,2,2025,323282.0,161641.0
2,3,2025,296242.0,49373.7
3,4,2025,191891.1,115134.7
4,5,2025,39561.3,19780.6
...,...,...,...,...
995,996,2025,308688.0,308688.0
996,997,2025,439038.9,274399.3
997,998,2025,102112.6,102112.6
998,999,2025,57003.4,28501.7


### Question 2
Find the 10 company-year records with the highest energy consumption.

Display:
- company_id
- year
- energy_consumption

In [10]:
run_sql("""
SELECT company_id , year , energy_consumption
FROM company_perf
ORDER BY energy_consumption DESC 
LIMIT 10
""" , connector = esg_lab_conn) 

,company_id,year,energy_consumption
0,759,2025,1.741047e+09
1,759,2024,1.569603e+09
2,759,2023,1.361814e+09
3,759,2022,1.162189e+09
4,646,2025,1.082296e+09
5,543,2025,1.009598e+09
6,646,2024,9.358958e+08
7,759,2021,9.149967e+08
8,646,2023,8.294755e+08
9,543,2024,8.152651e+08


### Question 3
Calculate the total carbon emissions for each year.

Display:
- year
- total carbon emissions

In [14]:
run_sql("""
SELECT year , SUM(carbon_emissions) as total_carbon_emissions
FROM company_perf
GROUP BY year
""" , connector = esg_lab_conn) 

,year,total_carbon_emissions
0,2015,9.922047e+08
1,2016,1.009856e+09
2,2017,1.058221e+09
3,2018,1.104560e+09
4,2019,1.159464e+09
5,2020,1.208687e+09
6,2021,1.271124e+09
7,2022,1.359086e+09
8,2023,1.467000e+09
9,2024,1.606672e+09


### Question 4
Calculate the average carbon emissions, water usage, and energy consumption for each industry

Display:
- industry
- average carbon emissions
- average water usage
- average energy consumption

In [18]:
## merge table (cuz industry is in company_info)
run_sql("""
SELECT  c.industry , 
        AVG(p.carbon_emissions) as avg_carbon_emissions , 
        AVG(p.water_usage) as avg_water_usage , 
        AVG(p.energy_consumption) as avg_energy_consumption
FROM company_perf as p 
JOIN company_info as c
    ON c.company_id = p.company_id
GROUP BY c.industry
""" , connector = esg_lab_conn) 

,industry,avg_carbon_emissions,avg_water_usage,avg_energy_consumption
0,Consumer Goods,3.403685e+05,4.254606e+05,1.701842e+06
1,Energy,8.191265e+06,2.457379e+06,8.191265e+07
2,Finance,7.205045e+04,3.602523e+04,1.801261e+05
3,Healthcare,3.353292e+05,3.353292e+05,6.706584e+05
4,Manufacturing,6.235968e+05,8.314625e+05,1.039328e+07
5,Retail,1.297126e+05,6.485629e+04,2.594252e+05
6,Technology,4.504187e+05,2.702512e+05,9.008375e+05
7,Transportation,6.503149e+05,1.083858e+05,2.167716e+06
8,Utilities,8.693411e+05,5.433382e+05,8.693411e+06


### Question 5
Find the 10 companies with the highest total carbon emissions.
Display:
- company_name
- industry
- total carbon emissions

In [20]:
run_sql("""
SELECT c.company_name , c.industry , SUM(p.carbon_emissions) AS total_carbon_emissions
FROM company_info as c 
JOIN company_perf as p
    ON c.company_id = p.company_id
GROUP BY c.company_name
ORDER BY total_carbon_emissions DESC
LIMIT 10
""" , connector = esg_lab_conn) 

,company_name,industry,total_carbon_emissions
0,Company_759,Energy,975938098.3
1,Company_646,Energy,750668362.9
2,Company_543,Energy,559408367.1
3,Company_535,Energy,363545497.2
4,Company_764,Energy,235433366.2
5,Company_651,Energy,225349323.6
6,Company_412,Energy,204529112.2
7,Company_285,Energy,194901190.0
8,Company_464,Energy,194091105.2
9,Company_98,Energy,171679455.9


### Question 6
Identify the industries with an average ESG environmental score (esg_environmental) greater than 65.

Display:
- industry
- average environmental ESG score
- average carbon emissions
- average energy consumption

In [22]:
run_sql("""
SELECT c.industry , AVG(p.esg_environmental) as average_esg , AVG(p.carbon_emissions) as average_carbon , AVG(p.energy_consumption) as average_energy
FROM company_info as c
JOIN company_perf as p
    ON c.company_id = p.company_id 
GROUP BY c.industry 
HAVING AVG(p.esg_environmental) > 65
""" , connector = esg_lab_conn) 

,industry,average_esg,average_carbon,average_energy
0,Finance,90.252856,72050.453982,180126.136766
1,Healthcare,66.228926,335329.185800,670658.371600
2,Technology,84.289761,450418.725837,900837.450622


### Question 7
Identify the 10 companies with the highest average
environmental ESG score.
Consider only performance records from 2020 to 2025.

Display:
- company_name
- industry
- average environmental ESG score
- average carbon emissions
- average water usage

In [26]:
run_sql("""
SELECT c.company_name , c.industry , AVG(p.esg_environmental) as average_esg , 
        AVG(p.carbon_emissions) as average_carbon , AVG(p.water_usage) as average_water

FROM company_info as c 
JOIN company_perf as p 
    ON c.company_id = p.company_id
    
WHERE p.year BETWEEN 2020 AND 2025
    
GROUP BY c.company_id
ORDER BY average_esg DESC  
LIMIT 10 
""" , connector = esg_lab_conn) 

,company_name,industry,average_esg,average_carbon,average_water
0,Company_972,Finance,100.000000,1.519727e+04,7.598633e+03
1,Company_920,Finance,100.000000,1.315193e+04,6.575967e+03
2,Company_731,Finance,100.000000,6.059375e+04,3.029687e+04
3,Company_441,Finance,100.000000,1.008355e+04,5.041800e+03
4,Company_307,Finance,100.000000,8.562462e+04,4.281230e+04
5,Company_123,Technology,100.000000,2.235865e+06,1.341519e+06
6,Company_107,Finance,100.000000,1.496602e+05,7.483008e+04
7,Company_27,Healthcare,100.000000,2.136062e+05,2.136062e+05
8,Company_4,Technology,100.000000,1.570584e+05,9.423505e+04
9,Company_509,Technology,99.983333,5.166713e+05,3.100028e+05
